# 문항 2: `in` 연산자 기반 타입 가드 분기 처리 및 제네릭 인메모리 저장소(Storage) 구축

## 2-1. 타입 정의 (01_data_modeling.ipynb에서 가져옴)

In [ ]:
type UserRole = 'admin' | 'customer' | 'seller';
type Category = 'electronics' | 'clothing' | 'food' | 'furniture' | 'books';

interface User {
  id: number;
  username: string;
  email: string;
  fullName: string;
  role: UserRole;
  createdAt: Date;
  phone?: string;
}

interface Product {
  id: number;
  title: string;
  description: string;
  price: number;
  category: Category;
  stock: number;
  isActive: boolean;
}

console.log('Types loaded.');

## 2-2. `in` 연산자 기반 타입 가드 함수

In [ ]:
// User | Product 유니언에서 in 연산자로 타입 좁히기(Narrowing)
const formatItem = (item: User | Product): string => {
  if ('email' in item) {
    // 참 분기: User 타입
    return `[회원] ${item.fullName} (${item.role})`;
  } else {
    // else 분기: Product 타입
    return `[상품] ${item.title} - ${item.price}원`;
  }
};

// 테스트 데이터
const sampleUser: User = {
  id: 1,
  username: 'test_user',
  email: 'test@example.com',
  fullName: '홍길동',
  role: 'customer',
  createdAt: new Date(),
};

const sampleProduct: Product = {
  id: 1,
  title: '노트북',
  description: '고성능 노트북',
  price: 1500000,
  category: 'electronics',
  stock: 10,
  isActive: true,
};

console.log(formatItem(sampleUser));
console.log(formatItem(sampleProduct));

## 2-3. 제네릭 인메모리 저장소(Storage) 구축

In [ ]:
// 제네릭 Storage 클래스 — id 필드가 있는 모든 타입에 재사용 가능
class InMemoryStorage<T extends { id: number }> {
  private items: T[] = [];

  add(item: T): void {
    this.items.push(item);
  }

  getAll(): T[] {
    return this.items;
  }

  getById(id: number): T | undefined {
    return this.items.find(item => item.id === id);
  }

  update(id: number, partial: Partial<T>): boolean {
    const index = this.items.findIndex(item => item.id === id);
    if (index === -1) return false;
    this.items[index] = { ...this.items[index], ...partial };
    return true;
  }

  delete(id: number): boolean {
    const before = this.items.length;
    this.items = this.items.filter(item => item.id !== id);
    return this.items.length < before;
  }

  count(): number {
    return this.items.length;
  }
}

console.log('InMemoryStorage class defined.');

## 2-4. 실행 확인

In [ ]:
// User Storage 사용
const userStore = new InMemoryStorage<User>();

userStore.add(sampleUser);
userStore.add({
  id: 2,
  username: 'admin',
  email: 'admin@example.com',
  fullName: '관리자',
  role: 'admin',
  createdAt: new Date(),
});

console.log('User count:', userStore.count());
console.log('User by id=1:', userStore.getById(1));

userStore.update(1, { role: 'admin' });
console.log('After update:', userStore.getById(1)?.role);

userStore.delete(2);
console.log('After delete, count:', userStore.count());

// Product Storage 사용 (동일 클래스 재사용)
const productStore = new InMemoryStorage<Product>();

productStore.add(sampleProduct);
console.log('Product count:', productStore.count());

// 타입 가드로 혼합 배열 처리
const mixed: (User | Product)[] = [sampleUser, sampleProduct];
mixed.forEach(item => console.log(formatItem(item)));